## Structured Output

Models can be requested to provide their response in a format matching a given schema.This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output .

## Pydantic 
Pydantic models provides the richest features set with field validations , descriptions and nested structures.

In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("Groq_Api_Key")

model = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0
)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.6', 'langchain': '1.4.9'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10fc175c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10fc17b90>, model_name='openai/gpt-oss-20b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [36]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie rating out of 10")
    
    

In [11]:
m = model.with_structured_output(
    Movie,
    method="json_schema"
)

result = m.invoke(
    "Provide details about the movie Gangs of Wasseypur"
)

print(result)

title='Gangs of Wasseypur' year=2012 director='Anurag Kashyap' rating=8.5


#### Messsage output alongside parsed structure  in Pydantic

In [14]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie rating out of 10")
m = model.with_structured_output(
    Movie,
    include_raw=True
)
result = m.invoke(
    "Provide details about the movie Gangs of Wasseypur"
)

print(result)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "We need to provide details about the movie Gangs of Wasseypur. The user didn't specify format. We can provide director, rating, title, year. We can use the function to get details. The function signature: Movie({director, rating, title, year}). We need to call the function. The user wants details about the movie. We can call the function with the known details: director: Anurag Kashyap, rating: maybe 8.5? But rating is out of 10. We can guess rating 8.5. Title: Gangs of Wasseypur. Year: 2012. Let's call the function.", 'tool_calls': [{'id': 'fc_69bc9555-7e94-4919-909f-297ea4723723', 'function': {'arguments': '{"director":"Anurag Kashyap","rating":8.5,"title":"Gangs of Wasseypur","year":2012}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 185, 'prompt_tokens': 163, 'total_tokens': 348, 'completion_time': 0.188892078, 'completion_tokens_details': {'reasoning_tokens':

#### Nested Structure in Pydantic

In [20]:
from pydantic import BaseModel , Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetail(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget in Millions USD")
m = model.with_structured_output(
    MovieDetail,
    method="json_schema"
)
result = m.invoke(
    "Provide details about the movie Gangs of Wasseypur"
)
print(result)


title='Gangs of Wasseypur' year=2012 cast=[Actor(name='Nawazuddin Siddiqui', role='Khan Saab'), Actor(name='Manoj Bajpayee', role='Bhiku Makhija'), Actor(name='Vijay Maurya', role='Sardar Khan'), Actor(name='Piyush Mishra', role='Javed Khan'), Actor(name='Sushant Singh', role="Khan Saab's son")] genres=['Crime', 'Drama', 'Thriller'] budget=1.5


## TypedDict

TypeDict provides a simple alternative using Python's built-in typing , ideal when you dont need runtime validations.

In [35]:
from typing_extensions import TypedDict , Annotated

class MovieDict(TypedDict):
    """A Movie With Details:"""
    title:Annotated[str,...,"The Titlr of the movie"]
    year:Annotated[int,...,"The year the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[float,...,"The Movie's rating out of 10"]

m=model.with_structured_output(MovieDict)
m.invoke("Please provide me the details of the web series Mirzapur")



{'director': 'Vikas Bahl, Vikramaditya Motwane',
 'rating': 8.5,
 'title': 'Mirzapur',
 'year': 2018}

## DataClasses 
A data class is a class typically mainly data although there aren't really any restrictions . You create it using the @dataclass decorator

In [ ]:
#pydantic
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_groq import ChatGroq

load_dotenv()

# 1. Define Actor schema
class Actor(BaseModel):
    name: str = Field(description="Actor's full name")
    role: str = Field(description="Character played by the actor")

# 2. Define Movie schema
class MovieDetail(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(
        default=None,
        description="Budget in millions of USD"
    )

# 3. Initialize Groq
model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("Groq_Api_Key"),
    temperature=0
)

# 4. Create agent
a = create_agent(
    model=model,
    tools=[],
    response_format=MovieDetail,
    system_prompt="""
    You are a movie information extraction assistant.
    Extract all available movie details.
    Never leave cast or genres empty when information is provided.
    Never invent unknown information.
    Use null for unknown budget.
    """
)

# 5. Invoke agent
r = a.invoke({
    "messages": [{
        "role": "user",
        "content": """
        Extract information about Saiyaara (2025).

        Movie information:
        - Ahaan Panday plays Krish Kapoor.
        - Aneet Padda plays Vaani Batra.
        - Genres: Romance and Drama.

        Return title, year, cast, genres and budget.
        Budget should be null if unknown.
        """
    }]
})

# 6. Print structured output
movie = r["structured_response"]

# json format output 
print(movie.model_dump_json(indent=2))

{
  "title": "Saiyaara",
  "year": 2025,
  "cast": [
    {
      "name": "Ahaan Panday",
      "role": "Krish Kapoor"
    },
    {
      "name": "Aneet Padda",
      "role": "Vaani Batra"
    }
  ],
  "genres": [
    "Romance",
    "Drama"
  ],
  "budget": null
}


In [52]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("Groq_Api_Key"),
    temperature=0
)

# 4. Create agent
a = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo,
)

result = a.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [ ]:
## Dataclass
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_groq import ChatGroq

load_dotenv()

@dataclass
class ContactInfo(BaseModel):
    name: str = Field(description="Provide your  full name")
    email: str=Field(description="Provide your email id ")
    phone: str=Field(description="Provide your Mobile Number")


# 3. Initialize Groq
model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("Groq_Api_Key"),
    temperature=0
)

# 4. Create agent
a = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo,
)

r=a.invoke({
    "messages":[{"role":"user", "content":"Extract contact info from: Ankit Kumar , ankit.iter07@gmail.com ,  +91 9031358194"}]
})
r["structured_response"]

ContactInfo(name='Ankit Kumar', email='ankit.iter07@gmail.com', phone='+91 9031358194')